In [ ]:
import lightkurve
import matplotlib.pyplot as plot
import numpy
from collections import defaultdict

%matplotlib inline
plot.style.use('seaborn-v0_8')

In [ ]:
'''
How to not have a million false positives!
1) Make sure that there is no v-shaped transit by zooming in
2) Make sure no even-odd rule
3) Remove data with noise to see others clearer
4) Even slight 2nd transits, like 1 / 10 the size of the initial transit, are EBs.
5) Ask ChatGPT about the period, first transit size and 2nd transit size: it tells you if it is bad.
6) Always look at the unflattened / untouched curves, but flatten if the fold cannot be made.
7) ALWAYS ZOOM IN to be sure about 2nd transits / v-shape
8) 2nd transits are not always at the 50% mark, and that can hint towards EB too.
9) Just because there is none of one doesn't mean there is none of the other
10) This is not a comprehensive method, but it gets rid of the false positives I can get rid of now.
11) Filter by radius / transit depth to find ones that seem not right but I didn't get rid of before
12) Adjust the periodogram if it isn't the right period. This happens when there is no noise removal, it should be fine including larger periods in regular usage.
13) MAKE SURE TO CHECK THE CENTROIDS
14) Even though EVERYTHING looks like it is a planet, simply too large of a radius indicates that it isn't.
15) Use more data if there is none (remove cadence = long, tess, etc.)
16) Check stellar radius --> high stellar radius means the v-shape is rounder for the same sized false positives
17) Check VSX, TEV, TESS EB
''';

In [ ]:
star_id = [
    219777401, # 0
    199710583, # 1
    198358825,
    20352516,
    238229705,
    467113954,
    297701617,
    73248090,
    279322914,
    38850860,
    238229705, # 10
    35655828,
    76903841,
    102625324,
    342556321,
    176871438,
    238229705,
    295440174,
    270462117,
    90083037,
    35655828, # 20
    734505581,
    11799872,
    11799874,
    55655482, # 24
    300291839,
    55727842,
    734505586,
    420114200,
    286883387,
    270404572,
    413162396,
    344175457,
    193271278,
    25776767,
    457599538,
    421191260,
    350335633,
    183513288,
    170112986,
    140167220,
    467447558,
    75048615,
    300291839,
    346347677,
    304369934,
    289713526,
    229452949,

    356107970,
    356086958,
    316910531,
    311188656,
    259171171,
    233603364,

    432114158,
    67512645,
    80439101,
    18373995,
    197807043,

    27847307,
    26748546,
    1715469667,
    441803068,
    41770205,
    420114774,

    352469839,
    22560356,
    435719275,
    62740096,

    140390696,
    470984719,

    70757284,
    365938088,

    167087044,
    141110149,

    452845139,
    4614514,

    270606344,
    270560025,
    258058036,
    182880367,

    5703872,
    459056711,
    290644615,

    25133286,

    359819459,
    308183430,

    142391477,
    358111274,

    92995573,
    165725822,
    167697604,

    235060821,
    257691369,

    384950919,
    390582796,
    398850090,
    422129452,

    451483379, # the radius is too big --> that's why Triceratops couldn't make a good fit except for EBs

    259019958,
    261401023,
    323350241, # right after is something that is about the same as the main one, and those disappear --> likely contamination since the small period disappears later

    462083435,

    30106140,
    405756510,
    410405725, # there is definitely something there but there is too much noise
    451112995,
    452522877,

    24440555,
    31368290,
    38462522,
    44988465,

    255989063
][-1] # -1 for last too

star_id = f'TIC {star_id}'
# star_id = 'HAT-P-11'
star_id

In [ ]:
custom_id = None
if custom_id is not None and custom_id != '':
    star_id = custom_id

star_id

In [ ]:
search_result = lightkurve.search_lightcurve(star_id, mission = 'TESS', cadence = 'long')
search_result

In [ ]:
sector_dict = defaultdict(list)

for entry in search_result:
    sector = entry.table['sequence_number'][0]
    sector_dict[sector].append(entry)

best_lightcurves = []
for sector, entries in sector_dict.items():
    spoc = None
    qlp = None

    for entry in entries:
        match entry.author:
            case 'SPOC' | 'TESS-SPOC' | 'TESS':
                spoc = entry

            case 'QLP':
                qlp = entry

    if spoc:  # spoc priority first
        best_lightcurves.append(spoc)

    elif qlp:
        best_lightcurves.append(qlp)

list(sector_dict.keys()), best_lightcurves

In [ ]:
collection = lightkurve.LightCurveCollection(None)
for entry in best_lightcurves[:10]:
    lc = entry.download().normalize()

    lc = lc[numpy.isfinite(lc.flux)]
    lc = lc[numpy.isfinite(lc.flux_err)]

    if lc.time.shape != (0,):
        lc.errorbar()
        collection.append(lc)

collection

In [ ]:
lc = collection[:].stitch().normalize()
lc = lc.flatten(window_length = 501, break_tolerance = 10, niters = 1, sigma = 10).remove_outliers(sigma = 10)
# lc = lc.remove_outliers(sigma = 10)

lc.errorbar()
# plot.gca().set_xlim(1667, 1667.4)
# plot.gca().set_ylim(0.994, 0.998)
plot.show()

In [ ]:
periods = numpy.logspace(0, 1.5, 10 ** 4, base = 10.0)
periodogram = lc.to_periodogram(method = 'bls', period = periods, frequency_factor = 10 ** 6)
periodogram.plot()
plot.show()

In [ ]:
period = periodogram.period_at_max_power
transit_time = periodogram.transit_time_at_max_power
transit_duration = periodogram.duration_at_max_power
period, transit_time, transit_duration

In [ ]:
# transit_time += period / 2 # 2nd eclipse
# period *= 2 # 2nd eclipse
# period /= 2 # wrong period
folded_lc = lc.fold(period = period, epoch_time = transit_time)
binned_lc = folded_lc.bin(time_bin_size = 0.01)
ax = folded_lc.errorbar()
binned_lc.plot(ax = ax, c = 'r')
d = 1
# ax.set_xlim(-d, d)
# ax.set_xlim(-3.4, -2.6)
# ax.set_ylim(0.99, 1.005)
plot.show()
transit_depth = 1 - numpy.nanmin(binned_lc.flux.value)
transit_depth